# 02 – Leakage Audit & Feature Decisions
Goal: check every feature for data leakage and decide which features the model may use.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.data import load_data

df = load_data("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.shape)
print(df["TotalCharges"].dtype)

(7043, 22)
float64


In [2]:
approx_total = df["tenure"] * df["MonthlyCharges"]
corr = df["TotalCharges"].corr(approx_total)
print(f"Correlation between TotalCharges and tenure × MonthlyCharges: {corr:.3f}")

Correlation between TotalCharges and tenure × MonthlyCharges: 1.000


In [3]:
from sklearn.metrics import roc_auc_score

y = df["churn_flag"]
results = {}

for col in df.columns.drop(["customerID", "Churn", "churn_flag"]):
    if df[col].dtype.kind in "if":
        score = df[col]
    else:
        score = df[col].map(df.groupby(col)["churn_flag"].mean())
    auc = roc_auc_score(y, score)
    results[col] = max(auc, 1 - auc)

pd.Series(results).sort_values(ascending=False).round(3)

tenure              0.740
Contract            0.739
OnlineSecurity      0.705
TechSupport         0.703
InternetService     0.695
OnlineBackup        0.678
PaymentMethod       0.671
DeviceProtection    0.671
TotalCharges        0.650
StreamingMovies     0.621
MonthlyCharges      0.621
StreamingTV         0.620
PaperlessBilling    0.607
Dependents          0.585
Partner             0.585
SeniorCitizen       0.563
MultipleLines       0.523
gender              0.505
PhoneService        0.504
dtype: float64

## Leakage review

| Feature | Known before churn? | Decision | Reason |
|---|---|---|---|
| customerID | Yes | ❌ Drop | Just an ID label, no predictive meaning; model could memorize it |
| Churn / churn_flag | — | 🎯 Target | This is what we predict |
| tenure | Yes (as of prediction date) | ✅ Keep | Strongest single predictor (AUC 0.740); the company always knows how long a customer has stayed |
| TotalCharges | Yes | ❌ Drop | Redundant: ≈ tenure × MonthlyCharges (correlation 1.000), adds no new information and is weaker alone (AUC 0.650) |
| gender | Yes | ❌ Drop | No predictive value (AUC 0.505 ≈ coin flip) and a fairness risk if used to decide who gets offers |
| Contract, PaymentMethod, InternetService, services | Yes (current account settings) | ✅ Keep | Strong signals in EDA (AUC 0.67–0.74), no leakage signs |
| MonthlyCharges, SeniorCitizen, Partner, Dependents, PaperlessBilling, PhoneService, MultipleLines | Yes | ✅ Keep | Known account/customer info; weak features alone may still help in combination |

**Leakage test:** no single feature has AUC > 0.95 (max = tenure, 0.740) → no obvious target leakage.

**Limitation:** the dataset is a single snapshot, so the exact time each value was recorded can't be verified.

In [4]:
from src.features import NUMERIC_FEATURES, CATEGORICAL_FEATURES, DROP_COLUMNS

features = NUMERIC_FEATURES + CATEGORICAL_FEATURES
print(f"Model will use {len(features)} features")

unaccounted = [c for c in df.columns if c not in features + DROP_COLUMNS]
print("Columns not accounted for:", unaccounted)

Model will use 17 features
Columns not accounted for: []
